In [ ]:
from google.colab import drive
drive.mount('/content/drive')

D = '/content/drive/MyDrive/connectome-project'
import sys; sys.path.insert(0, D)

import numpy as np, torch, os
from model import ConnectomeTransformer, mask_edges, masked_loss

print(torch.cuda.get_device_name(0))
print(torch.__version__)


Mounted at /content/drive
Tesla T4
2.11.0+cu128


In [ ]:
d  = np.load(f'{D}/scale33_normalized.npz')
sp = np.load(f'{D}/splits.npz')
X  = torch.tensor(d['X_log'], dtype=torch.float32)
N  = X.shape[1]
dev = 'cuda'

Xtr = X[sp['train']].to(dev)
Xva = X[sp['val']].to(dev)
Xte = X[sp['test']].to(dev)
print(f'train {tuple(Xtr.shape)}  val {tuple(Xva.shape)}  test {tuple(Xte.shape)}')

train (673, 73, 73)  val (144, 73, 73)  test (145, 73, 73)


In [ ]:
def train_one(use_id, seed, epochs=300, bs=64, lr=1e-3):
    torch.manual_seed(seed)
    m = ConnectomeTransformer(N, use_identity=use_id).to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    g = torch.Generator(device=dev).manual_seed(seed)
    best, best_sd, hist = float('inf'), None, []

    for ep in range(epochs):
        m.train()
        perm = torch.randperm(len(Xtr), device=dev)
        tl = 0.0
        for i in range(0, len(Xtr), bs):
            A = Xtr[perm[i:i+bs]]
            Am, msk = mask_edges(A, 0.15, g)
            pred, _ = m(Am)
            loss = masked_loss(pred, A, msk)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
            tl += loss.item() * len(A)
        sch.step()

        m.eval()
        with torch.no_grad():
            Am, msk = mask_edges(Xva, 0.15, g)
            vl = masked_loss(m(Am)[0], Xva, msk).item()
        hist.append((tl/len(Xtr), vl))
        if vl < best:
            best = vl
            best_sd = {k: v.clone() for k, v in m.state_dict().items()}

    m.load_state_dict(best_sd)
    return m, best, np.array(hist)


def extract(m):
    m.eval(); outs = []
    with torch.no_grad():
        for i in range(0, len(Xte), 32):
            _, at = m(Xte[i:i+32], return_attn=True)
            outs.append(torch.stack(at, dim=1).cpu())
    W = torch.cat(outs).numpy().astype(np.float32)
    assert np.allclose(W.sum(-1), 1.0, atol=1e-4), 'softmax rows broken'
    return W

print('ready')

ready


In [ ]:
for use_id, tag in [(False, 'A'), (True, 'B')]:
    for seed in [3, 4]:
        m, vl, hist = train_one(use_id, seed)
        np.save(f'{D}/attn_{tag}_seed{seed}.npy', extract(m))
        np.save(f'{D}/history_{tag}_seed{seed}.npy', hist)
        torch.save(m.state_dict(), f'{D}/model_{tag}_seed{seed}.pt')
        print(f'{tag} seed {seed}: val {vl:.4f}  saved')

A seed 3: val 0.4221  saved
A seed 4: val 0.4247  saved
B seed 3: val 0.3591  saved
B seed 4: val 0.3609  saved


In [ ]:
for use_id, tag in [(False, 'A'), (True, 'B')]:
    for seed in [1, 2, 3, 4]:
        torch.manual_seed(seed)
        m = ConnectomeTransformer(N, use_identity=use_id).to(dev)
        np.save(f'{D}/attn_ctrl{tag}_seed{seed}.npy', extract(m))
        print(f'ctrl{tag} seed {seed} saved')

ctrlA seed 1 saved
ctrlA seed 2 saved
ctrlA seed 3 saved
ctrlA seed 4 saved
ctrlB seed 1 saved
ctrlB seed 2 saved
ctrlB seed 3 saved
ctrlB seed 4 saved


In [ ]:
import glob
fs = sorted(glob.glob(f'{D}/attn_*.npy'))
print(f'{len(fs)} attention files\n')
for f in fs:
    a = np.load(f, mmap_mode='r')
    print(f'  {os.path.basename(f):28s} {str(a.shape):22s} {a.nbytes/1e6:.0f} MB')

vals = sorted(glob.glob(f'{D}/history_*.npy'))
print(f'\n{len(vals)} history files')
for f in vals:
    h = np.load(f)
    print(f'  {os.path.basename(f):28s} best val {h[:,1].min():.4f}')

20 attention files

  attn_A.npy                   (145, 2, 4, 73, 73)    25 MB
  attn_A_seed1.npy             (145, 2, 4, 73, 73)    25 MB
  attn_A_seed2.npy             (145, 2, 4, 73, 73)    25 MB
  attn_A_seed3.npy             (145, 2, 4, 73, 73)    25 MB
  attn_A_seed4.npy             (145, 2, 4, 73, 73)    25 MB
  attn_B.npy                   (145, 2, 4, 73, 73)    25 MB
  attn_B_seed1.npy             (145, 2, 4, 73, 73)    25 MB
  attn_B_seed2.npy             (145, 2, 4, 73, 73)    25 MB
  attn_B_seed3.npy             (145, 2, 4, 73, 73)    25 MB
  attn_B_seed4.npy             (145, 2, 4, 73, 73)    25 MB
  attn_ctrlA.npy               (145, 2, 4, 73, 73)    25 MB
  attn_ctrlA_seed1.npy         (145, 2, 4, 73, 73)    25 MB
  attn_ctrlA_seed2.npy         (145, 2, 4, 73, 73)    25 MB
  attn_ctrlA_seed3.npy         (145, 2, 4, 73, 73)    25 MB
  attn_ctrlA_seed4.npy         (145, 2, 4, 73, 73)    25 MB
  attn_ctrlB.npy               (145, 2, 4, 73, 73)    25 MB
  attn_ctrlB_seed1.n

In [ ]:
import itertools, numpy as np, torch

results = {}
for n_layers, seed in itertools.product([2, 4, 6], [0, 1, 2]):
    torch.manual_seed(seed)
    m = ConnectomeTransformer(N, n_layers=n_layers,
                              use_identity=False).to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, 300)
    g = torch.Generator(device=dev).manual_seed(seed)
    best, best_sd = float('inf'), None
    for ep in range(300):
        m.train()
        perm = torch.randperm(len(Xtr), device=dev)
        for i in range(0, len(Xtr), 64):
            A = Xtr[perm[i:i+64]]
            Am, msk = mask_edges(A, 0.15, g)
            loss = masked_loss(m(Am)[0], A, msk)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        sch.step(); m.eval()
        with torch.no_grad():
            Am, msk = mask_edges(Xva, 0.15, g)
            vl = masked_loss(m(Am)[0], Xva, msk).item()
        if vl < best:
            best = vl; best_sd = {k:v.clone() for k,v in m.state_dict().items()}
    m.load_state_dict(best_sd)
    np.save(f'{D}/attn_depth{n_layers}_seed{seed}.npy', extract(m))
    print(f'layers {n_layers} seed {seed}: val {best:.4f}')

layers 2 seed 0: val 0.4260
layers 2 seed 1: val 0.4292
layers 2 seed 2: val 0.4231
layers 4 seed 0: val 0.3875
layers 4 seed 1: val 0.3847
layers 4 seed 2: val 0.3869
layers 6 seed 0: val 0.3757
layers 6 seed 1: val 0.3745
layers 6 seed 2: val 0.3791


In [ ]:
for use_id, tag in [(False, 'A'), (True, 'B')]:
    for seed in [1, 2, 5, 6, 7, 8, 9]:
        m, vl, hist = train_one(use_id, seed)
        np.save(f'{D}/attn_{tag}_seed{seed}.npy', extract(m))
        np.save(f'{D}/history_{tag}_seed{seed}.npy', hist)
        print(f'{tag} seed {seed}: val {vl:.4f}')

for use_id, tag in [(False, 'A'), (True, 'B')]:
    for seed in [1, 2, 5, 6, 7, 8, 9]:
        torch.manual_seed(seed)
        np.save(f'{D}/attn_ctrl{tag}_seed{seed}.npy',
                extract(ConnectomeTransformer(N, use_identity=use_id).to(dev)))
print('done')

A seed 1: val 0.4292
A seed 2: val 0.4231
A seed 5: val 0.4307
A seed 6: val 0.4251
A seed 7: val 0.4213
A seed 8: val 0.4246
A seed 9: val 0.4239
B seed 1: val 0.3601
B seed 2: val 0.3526
B seed 5: val 0.3561
B seed 6: val 0.3605
B seed 7: val 0.3597
B seed 8: val 0.3636
B seed 9: val 0.3604
done


In [ ]:
import numpy as np, glob, os
d = "/content/drive/MyDrive/<your folder>"
for f in ["attn_A.npy", "attn_A_seed1.npy", "attn_ctrlA_seed1.npy",
          "attn_depth2_seed0.npy", "history_A.npy", "test_idx.npy"]:
    a = np.load(os.path.join(d, f), allow_pickle=True)
    print(f, a.shape, a.dtype)
print(np.load(os.path.join(d, "splits.npz")).files)
print(glob.glob(os.path.join(d, "scale33*")))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/connectome-project/attn_A.npy'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')